In [ ]:
import sys
!{sys.executable} -m pip uninstall -y numpy
!{sys.executable} -m pip install "numpy<2"

In [ ]:
import pandas as pd
import numpy as np

# Load the Excel file
xls = pd.ExcelFile(r'D:\Downloads\DATA_DRYAD__Roy_et_al._2020.xlsx')
df_sheet2 = pd.read_excel(xls, sheet_name='Sheet 2', header=None)

def col_letter_to_index(letter):
    return ord(letter.upper()) - ord('A')

rows = []

for colour, (start_col, end_col, start_row, end_row) in colour_blocks.items():

    long_col = col_letter_to_index(start_col)
    short_col = long_col + 1

    for r in range(start_row - 1, end_row):

        length = df_sheet2.iloc[r, long_col]
        width = df_sheet2.iloc[r, short_col]

        if pd.notna(length) and pd.notna(width) and width != 0:

            rows.append([
                length,
                width,
                length / width,
                colour
            ])
df_train = pd.DataFrame(rows, columns=['length_nm', 'width_nm', 'aspect_ratio', 'colour'])
df_train.to_csv('modern_bird_training.csv', index=False)

In [ ]:
# Manually input measurements from Sheet 4
theropod_data = [
    # Microraptor (iridescent)
    [1157, 230, 1157/230, 'iridescent'],
    # Anchiornis elongate (black)
    [1079, 293, 1079/293, 'black'],
    # Anchiornis oblong (grey/brown)
    [854, 563, 854/563, 'brown'],
    # ... add all
]
df_val = pd.DataFrame(theropod_data, columns=['length_nm','width_nm','aspect_ratio','known_colour'])

In [ ]:
print(df_val)

In [ ]:
import pandas as pd
import numpy as np

file_path = r"D:\Downloads\rsos251232_si_002.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1")

# ---- 1. Training set: all rows with a known Colour (not NaN) ----
modern_df = df[df["Colour"].notna()].copy()
modern_df = modern_df.rename(columns={
    "Melanosomes": "species",
    "diameter1(>micro)": "length_nm",
    "diameter2": "width_nm",
    "Aspect Ratio": "aspect_ratio"
})
modern_df = modern_df[["species", "Colour", "length_nm", "width_nm", "aspect_ratio"]].dropna()
modern_df.to_csv("modern_training.csv", index=False)
print(f"Training set saved: {len(modern_df)} samples")

# ---- 2. Fossil rows: Colour is NaN ----
fossil_df = df[df["Colour"].isna()].copy()
fossil_df = fossil_df.rename(columns={
    "Melanosomes": "species",
    "diameter1(>micro)": "length_nm",
    "diameter2": "width_nm",
    "Aspect Ratio": "aspect_ratio"
})

# ---- 3. Theropod validation set (by species names) ----
theropod_names = [
    "Anchiornis huxleyi", "Archaeopteryx lithographica", "Caudipteryx zoui",
    "Microraptor sp.", "Sinosauropteryx", "Wulong sp. (ulna)",
    "Wulong sp. (ilium)", "Wulong sp. (abdomen)"
]
theropod_df = fossil_df[fossil_df["species"].isin(theropod_names)].copy()
colour_map = {
    "Anchiornis huxleyi": "black",
    "Archaeopteryx lithographica": "black",
    "Caudipteryx zoui": "black",
    "Microraptor sp.": "iridescent",
    "Sinosauropteryx": "brown",
    "Wulong sp. (ulna)": "black",
    "Wulong sp. (ilium)": "black",
    "Wulong sp. (abdomen)": "grey"
}
theropod_df["known_colour"] = theropod_df["species"].map(colour_map)
theropod_df = theropod_df[["species", "known_colour", "length_nm", "width_nm", "aspect_ratio"]].dropna()
theropod_df.to_csv("theropod_validation.csv", index=False)
print(f"Theropod validation set saved: {len(theropod_df)} samples")

# ---- 4. Non‑theropod test set (Psittacosaurus, Diplodocus) ----
non_theropod_names = [
    "Psittacosaurus lujiatunensis", "Diplodocus sp. 10660", "Diplodocus 10662"
]
test_df = fossil_df[fossil_df["species"].isin(non_theropod_names)].copy()
test_df = test_df[["species", "length_nm", "width_nm", "aspect_ratio"]].dropna()
test_df.to_csv("non_theropod_test.csv", index=False)
print(f"Non-theropod test set saved: {len(test_df)} samples")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---- Load data ----
train = pd.read_csv("modern_training.csv")
val = pd.read_csv("theropod_validation.csv")
test = pd.read_csv("non_theropod_test.csv")

# ---- FIX: convert all colour strings to lowercase ----
train["Colour"] = train["Colour"].str.lower()
val["known_colour"] = val["known_colour"].str.lower()

print("Training set class distribution:")
print(train["Colour"].value_counts())

# ---- Encode colours ----
le = LabelEncoder()
train["colour_enc"] = le.fit_transform(train["Colour"])
val["colour_enc"] = le.transform(val["known_colour"])

# Features
features = ["length_nm", "width_nm", "aspect_ratio"]
X_train = train[features]
y_train = train["colour_enc"]
X_val = val[features]
y_val = val["colour_enc"]
X_test = test[features]

# Standardise
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ---- Random Forest ----
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"\nRF 5‑fold CV accuracy: {rf_cv.mean():.3f} (+/- {rf_cv.std():.3f})")

val_pred_rf = rf.predict(X_val_scaled)
print(f"RF theropod validation accuracy: {accuracy_score(y_val, val_pred_rf):.3f}")
print("Classification report (RF on theropods):")
print(classification_report(y_val, val_pred_rf, target_names=le.classes_))

# Predict non‑theropods
test_pred_rf = rf.predict(X_test_scaled)
test_proba_rf = rf.predict_proba(X_test_scaled)
print("\nRF predictions for non‑theropods:")
for i, row in test.iterrows():
    pred = le.inverse_transform([test_pred_rf[i]])[0]
    prob = max(test_proba_rf[i])
    print(f"  {row['species']}: {pred} (prob={prob:.2f})")

# ---- Support Vector Machine ----
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

svm_cv = cross_val_score(svm, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"\nSVM 5‑fold CV accuracy: {svm_cv.mean():.3f} (+/- {svm_cv.std():.3f})")

val_pred_svm = svm.predict(X_val_scaled)
print(f"SVM theropod validation accuracy: {accuracy_score(y_val, val_pred_svm):.3f}")

test_pred_svm = svm.predict(X_test_scaled)
test_proba_svm = svm.predict_proba(X_test_scaled)
print("\nSVM predictions for non‑theropods:")
for i, row in test.iterrows():
    pred = le.inverse_transform([test_pred_svm[i]])[0]
    prob = max(test_proba_svm[i])
    print(f"  {row['species']}: {pred} (prob={prob:.2f})")

# ---- Feature importance (RF) ----
importances = rf.feature_importances_
print("\nRF feature importances:")
for f, imp in zip(features, importances):
    print(f"  {f}: {imp:.3f}")

In [ ]:
"""
Complete corrected script for dinosaur colour prediction.
- Converts all measurements to nanometres (nm).
- Combines training data from both provided CSV files.
- Trains Random Forest and SVM.
- Validates on theropod fossils.
- Predicts colour for non‑theropod dinosaurs.
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

# ------------------------------------------------------------
# 1. Load and prepare training data
# ------------------------------------------------------------

# ---- Training set 1: modern_training.csv (already in nm) ----
train1 = pd.read_csv("modern_training.csv")
# Keep only necessary columns and standardise colour names
train1 = train1[["Colour", "length_nm", "width_nm", "aspect_ratio"]].copy()
train1["Colour"] = train1["Colour"].str.lower()
train1.rename(columns={"Colour": "colour"}, inplace=True)

# ---- Training set 2: modern_bird_training.csv (RAW values are in µm!) ----
train2_raw = pd.read_csv("modern_bird_training.csv")
# Multiply length and width by 1000 to convert µm → nm
train2_raw["length_nm"] = train2_raw["length_nm"] * 1000
train2_raw["width_nm"]  = train2_raw["width_nm"] * 1000
train2_raw["aspect_ratio"] = train2_raw["length_nm"] / train2_raw["width_nm"]
train2 = train2_raw[["colour", "length_nm", "width_nm", "aspect_ratio"]].copy()
train2["colour"] = train2["colour"].str.lower()

# ---- Combine both training sets ----
train = pd.concat([train1, train2], ignore_index=True)
print(f"Total training samples after unit correction: {len(train)}")
print("Class distribution:")
print(train["colour"].value_counts())

# ------------------------------------------------------------
# 2. Load validation and test sets (both already in nm)
# ------------------------------------------------------------
val = pd.read_csv("theropod_validation.csv")
val["known_colour"] = val["known_colour"].str.lower()

test = pd.read_csv("non_theropod_test.csv")

# ------------------------------------------------------------
# Feature engineering (ADD THIS)
# ------------------------------------------------------------
def engineer_dino_features(df):
    df = df.copy()

    df['volume_proxy'] = (
        (4/3) * np.pi *
        (df['length_nm'] / 2) *
        ((df['width_nm'] / 2) ** 2)
    )

    df['log_length'] = np.log1p(df['length_nm'])
    df['log_width']  = np.log1p(df['width_nm'])

    return df

train = engineer_dino_features(train)
val   = engineer_dino_features(val)
test  = engineer_dino_features(test)

# ------------------------------------------------------------
# 3. Encode colour labels
# ------------------------------------------------------------
le = LabelEncoder()
train["colour_enc"] = le.fit_transform(train["colour"])
val["colour_enc"]   = le.transform(val["known_colour"])

# Feature columns
features = ["aspect_ratio", "volume_proxy", "log_length", "log_width"]
X_train = train[features]
y_train = train["colour_enc"]
X_val   = val[features]
y_val   = val["colour_enc"]
X_test  = test[features]

# ------------------------------------------------------------
# 4. Standardise features (important for SVM, also helps RF)
# ------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# ------------------------------------------------------------
# 5. Random Forest
# ------------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=5,
    class_weight='balanced_subsample',
    random_state=42
)
rf.fit(X_train_scaled, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print("\n" + "="*50)
print("RANDOM FOREST")
print(f"5‑fold CV accuracy: {rf_cv_scores.mean():.3f} (+/- {rf_cv_scores.std():.3f})")

val_pred_rf = rf.predict(X_val_scaled)
print(f"Theropod validation accuracy: {accuracy_score(y_val, val_pred_rf):.3f}")
print("\nClassification report (theropod validation):")
print(classification_report(y_val, val_pred_rf, target_names=le.classes_, zero_division=0))

# Predict non‑theropods
test_pred_rf = rf.predict(X_test_scaled)
test_proba_rf = rf.predict_proba(X_test_scaled)
print("\nPredictions for non‑theropods (RF):")
for i, row in test.iterrows():
    pred_colour = le.inverse_transform([test_pred_rf[i]])[0]
    prob = max(test_proba_rf[i])
    print(f"  {row['species']}: {pred_colour} (prob={prob:.2f})")

# Feature importances
print("\nRF feature importances:")
for f, imp in zip(features, rf.feature_importances_):
    print(f"  {f}: {imp:.3f}")

# ------------------------------------------------------------
# 6. Support Vector Machine
# ------------------------------------------------------------
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)
svm.fit(X_train_scaled, y_train)

svm_cv_scores = cross_val_score(svm, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print("\n" + "="*50)
print("SUPPORT VECTOR MACHINE")
print(f"5‑fold CV accuracy: {svm_cv_scores.mean():.3f} (+/- {svm_cv_scores.std():.3f})")

val_pred_svm = svm.predict(X_val_scaled)
print(f"Theropod validation accuracy: {accuracy_score(y_val, val_pred_svm):.3f}")

test_pred_svm = svm.predict(X_test_scaled)
test_proba_svm = svm.predict_proba(X_test_scaled)
print("\nPredictions for non‑theropods (SVM):")
for i, row in test.iterrows():
    pred_colour = le.inverse_transform([test_pred_svm[i]])[0]
    prob = max(test_proba_svm[i])
    print(f"  {row['species']}: {pred_colour} (prob={prob:.2f})")

# ------------------------------------------------------------
# End of script
# ------------------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# Load and prepare data (same as before, but remove penguin)
train1 = pd.read_csv("modern_training.csv")
train1 = train1[["Colour", "length_nm", "width_nm", "aspect_ratio"]].copy()
train1["Colour"] = train1["Colour"].str.lower()
train1.rename(columns={"Colour": "colour"}, inplace=True)

train2_raw = pd.read_csv("modern_bird_training.csv")
train2_raw["length_nm"] = train2_raw["length_nm"] * 1000
train2_raw["width_nm"] = train2_raw["width_nm"] * 1000
train2_raw["aspect_ratio"] = train2_raw["length_nm"] / train2_raw["width_nm"]
train2 = train2_raw[["colour", "length_nm", "width_nm", "aspect_ratio"]].copy()
train2["colour"] = train2["colour"].str.lower()

train = pd.concat([train1, train2], ignore_index=True)
# Remove penguin – not relevant to dinosaurs
train = train[train['colour'] != 'penguin']

# Simple features: aspect_ratio only (most robust to taphonomy)
features = ['aspect_ratio']
X_train = train[features]
y_train = train['colour']

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Balance with SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train_enc)

# Load validation (theropod)
val = pd.read_csv("theropod_validation.csv")
val["known_colour"] = val["known_colour"].str.lower()
X_val = val[features]
y_val = val["known_colour"]
y_val_enc = le.transform(y_val)
X_val_scaled = scaler.transform(X_val)

# Load test (non-theropod)
test = pd.read_csv("non_theropod_test.csv")
X_test = test[features]
X_test_scaled = scaler.transform(X_test)

# Models
models = {
    'QDA': QuadraticDiscriminantAnalysis(),
    'SVM': SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=42),
    'RF': RandomForestClassifier(n_estimators=150, max_depth=5, class_weight='balanced_subsample', random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n{'='*40}\n{name}")
    # Handle QDA separately (no probability)
    if name == 'QDA':
        model.fit(X_train_bal, y_train_bal)
        cv_scores = cross_val_score(model, X_train_bal, y_train_bal, cv=cv, scoring='accuracy')
        print(f"5‑fold CV: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
        val_pred = model.predict(X_val_scaled)
        print(f"Theropod validation accuracy: {accuracy_score(y_val_enc, val_pred):.3f}")
        # After val_pred = model.predict(X_val_scaled) for each model
        print("\nIndividual theropod predictions:")
        for i, (idx, row) in enumerate(val.iterrows()):
            true_color = row['known_colour']
            pred_color = le.inverse_transform([val_pred[i]])[0]
            correct = "✓" if true_color == pred_color else "✗"
            print(f"  {row['species']:25s} | true: {true_color:10s} | pred: {pred_color:10s} | {correct}")
        # Non-theropod predictions
        test_pred = model.predict(X_test_scaled)
        print("Non‑theropod predictions:")
        for i, row in test.iterrows():
            print(f"  {row['species']}: {le.inverse_transform([test_pred[i]])[0]}")
    else:
        model.fit(X_train_bal, y_train_bal)
        cv_scores = cross_val_score(model, X_train_bal, y_train_bal, cv=cv, scoring='accuracy')
        print(f"5‑fold CV: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
        val_pred = model.predict(X_val_scaled)
        print(f"Theropod validation accuracy: {accuracy_score(y_val_enc, val_pred):.3f}")
        print("\nIndividual theropod predictions:")
        for i, (idx, row) in enumerate(val.iterrows()):
            true_color = row['known_colour']
            pred_color = le.inverse_transform([val_pred[i]])[0]
            correct = "✓" if true_color == pred_color else "✗"
            print(f"  {row['species']:25s} | true: {true_color:10s} | pred: {pred_color:10s} | {correct}")
        test_pred = model.predict(X_test_scaled)
        test_proba = model.predict_proba(X_test_scaled)
        print("Non‑theropod predictions:")
        for i, row in test.iterrows():
            pred = le.inverse_transform([test_pred[i]])[0]
            prob = max(test_proba[i])
            print(f"  {row['species']}: {pred} (conf={prob:.2f})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use only the four main colour classes (exclude 'penguin' if present)
plot_data = train[train['colour'].isin(['black', 'brown', 'grey', 'iridescent'])].copy()

# Order boxes by median aspect ratio (optional)
order = plot_data.groupby('colour')['aspect_ratio'].median().sort_values().index

plt.figure(figsize=(8, 5))
sns.boxplot(data=plot_data, x='colour', y='aspect_ratio', order=order,
            palette='Set2', showfliers=False)

# Add swarm plot to show individual points (helps see density)
sns.stripplot(data=plot_data, x='colour', y='aspect_ratio', order=order,
              color='black', alpha=0.3, size=2)

plt.ylabel('Aspect Ratio (Length / Width)', fontsize=12)
plt.xlabel('Colour Class', fontsize=12)
plt.title('Melanosome Aspect Ratio by Color in Modern Birds', fontsize=14)

# Save high-resolution figure
plt.tight_layout()
plt.savefig('figure1_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use log scale to spread the data
train_plot = train[train['colour'].isin(['black', 'brown', 'grey', 'iridescent'])].copy()
train_plot['log_length'] = np.log10(train_plot['length_nm'])
train_plot['log_width']  = np.log10(train_plot['width_nm'])

# Define colours for each class
colour_palette = {'black': '#2c3e50', 'brown': '#8B4513', 
                  'grey': '#7f8c8d', 'iridescent': '#1abc9c'}

fig, ax = plt.subplots(figsize=(8, 6))

# Plot training data
for colour, group in train_plot.groupby('colour'):
    ax.scatter(group['log_length'], group['log_width'], 
               label=colour.capitalize(), color=colour_palette[colour],
               alpha=0.5, s=20, edgecolor='none')

# Overlay non‑theropod fossils with stars
test_plot = test.copy()
test_plot['log_length'] = np.log10(test_plot['length_nm'])
test_plot['log_width']  = np.log10(test_plot['width_nm'])

# Map species to labels for legend
species_labels = {
    'Psittacosaurus lujiatunensis': 'Psittacosaurus',
    'Diplodocus sp. 10660': 'Diplodocus',
    'Diplodocus 10662': 'Diplodocus'
}
for idx, row in test_plot.iterrows():
    label = species_labels.get(row['species'], row['species'])
    ax.scatter(row['log_length'], row['log_width'], 
               marker='*', s=200, color='red', 
               edgecolor='black', linewidth=0.5,
               label=label if idx == test_plot.index[0] else "")

ax.set_xlabel('log₁₀(Length / nm)', fontsize=12)
ax.set_ylabel('log₁₀(Width / nm)', fontsize=12)
ax.set_title('Melanosome Morphospace: Training vs. Non‑Theropod Fossils', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('figure2_distribution_shift.png', dpi=300, bbox_inches='tight')
plt.show()